## Build pickle file for SMARTBind model training

In [ ]:
import pickle
from pathlib import Path


def find_repo_root(start):
    for base in (Path(start).resolve(), *Path(start).resolve().parents):
        if (base / "smartbind" / "__init__.py").is_file():
            return base
    raise FileNotFoundError("Run this notebook from inside a checkout of the repository.")


# A shipped training pickle, used here only as a worked example of the schema. Point this
# at your own file once you have built one.
REPO = find_repo_root(Path.cwd())
with open(REPO / "data" / "hariboss" / "hariboss_10fd_with_decoys.pkl", "rb") as f:
    data = pickle.load(f)


The python dictionary structure should be as follows:
- `data.keys()`: RNA-ligand complex id;
- `data['3ds7'].keys()`: id for specific RNA-ligand pairs in a single PDB, the different formating of this id name will not affect the processing pipeline.

Here is example:

In [5]:
list(data.keys())[0:5]

['1aju', '1akx', '1am0', '1arj', '1eht']

In [7]:
data['3ds7'].keys()

dict_keys(['3ds7_A_I_GNG_120', '3ds7_B_Q_GNG_320'])

In [8]:
data['3ds7']['3ds7_B_Q_GNG_320']['train_split']

[True, True, True, True, True, True, True, False, True, True]

The minimum information requirements for `data['3ds7']['3ds7_B_Q_GNG_320'].keys()` (based on example above):
- 'pdb_id': '3ds7'
- 'train_split': [True, False, True]  --> this key is used for fold-splitting. If the `fold_num` in `binding_training.yaml` file is set to 1, then the pairs will be assigned into training dataset if the first element in the `train_split`  is True, otherwise it will be assigned into the validation dataset.
- 'rna_chain_id': 'B'
- 'rna_chain_sequence': 'GGACAUACAAUCGCGUGGAUAUGGCACGCAAGAUCCCGCCGGGCACCGUAAAUGUCCGACUAUGUCC' 
- 'downloaded_ligand_smiles': 'Nc1nc(=O)c2ncn([C@H]3C[C@H](O)[C@@H](CO)O3)c2[nH]1'
- 'fp2': [0, 0, 0, 0, 0, ..., 1, 0] --> 1024-dimension list, `FP2(downloaded_ligand_smiles)`; build it with `smartbind.preprocess.convert_smiles_to_pf2`
- 'ligand_id': 'GNG'
- 'contact_map_1d': [0, 1, 1, 0, 0, ..., 1, 0] --> binary groundtruth of the binding site in list, the length should be equal to the length of RNA chain sequence.
- 'decoy_smiles': augmented SMILES sampled by ligand-specific decoy augmentation strategy (see `augmentation_data.ipynb`).
- 'decoy_fp2': the FP2 fingerprints of those decoys, in the same order --> shape (n_decoys, 1024)

`fp2` and `decoy_fp2` are the two keys `smartbind/preprocess.py` reads directly, so a pickle
missing either of them fails at load with a `KeyError`. Note that files from before this
revision spelled `fp2` as `download_fp2` and stored fingerprints (not SMILES) under
`decoy_smiles`; those are not readable by the current code.

Save the finished pickle wherever you like -- `data/hariboss/` alongside the shipped ones is
the natural place -- then point `data_path` in the yaml configuration at it for training.
